# 🔢 Day 02a: Subnetting — CIDR, VLSM & Subnet Calculator

---

## 🎯 What You'll Master Today
- Subnet calculations step-by-step
- VLSM (Variable Length Subnet Masking)
- Supernetting / aggregation
- Interview-speed subnet tricks

---

### 🎭 The Analogy

Subnetting = dividing a **large plot of land** into smaller fenced-off sections.  
Each fence (subnet mask) decides how many houses (hosts) fit inside.

```
╔══════════════════════════════════════════════════════════════════════╗
║              SUBNETTING CHEAT SHEET                                 ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  /24 → 256 addresses → 254 usable │ mask 255.255.255.0             ║
║  /25 → 128 addresses → 126 usable │ mask 255.255.255.128           ║
║  /26 →  64 addresses →  62 usable │ mask 255.255.255.192           ║
║  /27 →  32 addresses →  30 usable │ mask 255.255.255.224           ║
║  /28 →  16 addresses →  14 usable │ mask 255.255.255.240           ║
║  /29 →   8 addresses →   6 usable │ mask 255.255.255.248           ║
║  /30 →   4 addresses →   2 usable │ mask 255.255.255.252           ║
║  /31 →   2 addresses →   2 usable │ point-to-point links           ║
║  /32 →   1 address   →   1 (host) │ single host route              ║
║                                                                      ║
║  Magic Number = 256 - last non-zero mask octet                      ║
║  E.g., mask 255.255.255.192 → magic = 256-192 = 64                 ║
║  Subnets: .0, .64, .128, .192 (jump by magic number)               ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. Full Subnet Calculator
# ============================================

class SubnetCalculator:
    def __init__(self, cidr):
        ip, prefix = cidr.split('/')
        self.ip = ip
        self.prefix = int(prefix)
        self.ip_int = self._ip_to_int(ip)
        self.mask_int = (0xFFFFFFFF << (32 - self.prefix)) & 0xFFFFFFFF
    
    def _ip_to_int(self, ip):
        octets = list(map(int, ip.split('.')))
        return sum(o << (8 * (3 - i)) for i, o in enumerate(octets))
    
    def _int_to_ip(self, num):
        return '.'.join(str((num >> (8 * i)) & 0xFF) for i in range(3, -1, -1))
    
    def network(self):    return self._int_to_ip(self.ip_int & self.mask_int)
    def broadcast(self):  return self._int_to_ip(self.ip_int | ~self.mask_int & 0xFFFFFFFF)
    def first_host(self): return self._int_to_ip((self.ip_int & self.mask_int) + 1)
    def last_host(self):  return self._int_to_ip((self.ip_int | ~self.mask_int & 0xFFFFFFFF) - 1)
    def mask(self):       return self._int_to_ip(self.mask_int)
    def wildcard(self):   return self._int_to_ip(~self.mask_int & 0xFFFFFFFF)
    def total(self):      return 2 ** (32 - self.prefix)
    def usable(self):     return max(self.total() - 2, 0)
    
    def report(self):
        print(f"  ═══ Subnet Report: {self.ip}/{self.prefix} ═══\n")
        print(f"    Network Address  : {self.network()}")
        print(f"    Subnet Mask      : {self.mask()}")
        print(f"    Wildcard Mask    : {self.wildcard()}")
        print(f"    First Usable     : {self.first_host()}")
        print(f"    Last Usable      : {self.last_host()}")
        print(f"    Broadcast        : {self.broadcast()}")
        print(f"    Total Addresses  : {self.total():,}")
        print(f"    Usable Hosts     : {self.usable():,}")
        
        # Magic number
        mask_octets = list(map(int, self.mask().split('.')))
        last_nonzero = next((o for o in reversed(mask_octets) if o != 0 and o != 255), None)
        if last_nonzero:
            print(f"    Magic Number     : {256 - last_nonzero}")

# Interactive examples
for cidr in ['192.168.1.130/26', '10.0.0.0/12', '172.16.50.0/28']:
    SubnetCalculator(cidr).report()
    print()

In [ ]:
# ============================================
# 2. VLSM — Variable Length Subnet Masking
# ============================================
import math

def vlsm_allocate(network_cidr, requirements):
    """
    Given a network and a list of (name, host_count) requirements,
    allocate subnets using VLSM (largest first).
    """
    ip, prefix = network_cidr.split('/')
    prefix = int(prefix)
    ip_int = sum(o << (8 * (3 - i)) for i, o in enumerate(map(int, ip.split('.'))))
    network_start = ip_int & ((0xFFFFFFFF << (32 - prefix)) & 0xFFFFFFFF)
    
    # Sort by need (largest first for efficiency)
    sorted_reqs = sorted(requirements, key=lambda x: x[1], reverse=True)
    
    current = network_start
    subnets = []
    
    print(f"  ═══ VLSM Allocation for {network_cidr} ═══\n")
    print(f"  {'Subnet':>12} {'Hosts Needed':>13} {'Prefix':>7} {'Block Size':>11} {'Network':>18}")
    print(f"  {'─'*12} {'─'*13} {'─'*7} {'─'*11} {'─'*18}")
    
    for name, hosts_needed in sorted_reqs:
        # Need hosts + network + broadcast
        total_needed = hosts_needed + 2
        # Find smallest power of 2 >= total_needed
        bits_host = math.ceil(math.log2(total_needed))
        block_size = 2 ** bits_host
        new_prefix = 32 - bits_host
        
        # Align to block boundary
        if current % block_size != 0:
            current = ((current // block_size) + 1) * block_size
        
        net_addr = '.'.join(str((current >> (8 * i)) & 0xFF) for i in range(3, -1, -1))
        
        print(f"  {name:>12} {hosts_needed:>13} {'/' + str(new_prefix):>7} {block_size:>11} {net_addr + '/' + str(new_prefix):>18}")
        
        subnets.append((name, net_addr, new_prefix, block_size))
        current += block_size
    
    return subnets

# Typical interview problem
vlsm_allocate('192.168.1.0/24', [
    ('Sales',     60),
    ('Engineering', 28),
    ('HR',         12),
    ('Management',  5),
    ('PointToPoint', 2),
])

In [ ]:
# ============================================
# 3. Supernetting / Route Aggregation
# ============================================

def supernet(networks):
    """Find the smallest supernet covering all listed networks."""
    def ip_to_int(ip):
        return sum(int(o) << (8 * (3 - i)) for i, o in enumerate(ip.split('.')))
    
    def int_to_ip(n):
        return '.'.join(str((n >> (8 * i)) & 0xFF) for i in range(3, -1, -1))
    
    ints = [ip_to_int(n.split('/')[0]) for n in networks]
    min_ip = min(ints)
    max_ip = max(ints)
    
    # Find common prefix bits
    xor = min_ip ^ max_ip
    common_bits = 0
    for i in range(31, -1, -1):
        if xor & (1 << i):
            break
        common_bits += 1
    
    mask = (0xFFFFFFFF << (32 - common_bits)) & 0xFFFFFFFF
    supernet_addr = min_ip & mask
    
    print(f"  ═══ Route Aggregation ═══\n")
    print(f"    Input networks:")
    for n in networks:
        print(f"      {n}")
    print(f"\n    Supernet: {int_to_ip(supernet_addr)}/{common_bits}")
    print(f"    (Reduces {len(networks)} routes → 1 route)")

# Example: aggregate contiguous /24s
supernet(['192.168.0.0/24', '192.168.1.0/24', '192.168.2.0/24', '192.168.3.0/24'])

In [ ]:
# ============================================
# 4. Subnetting Speed Drill
# ============================================

print("""
  ╔══════════════════════════════════════════════════════════════════╗
  ║               SUBNETTING SPEED TRICKS (Interview)              ║
  ╠══════════════════════════════════════════════════════════════════╣
  ║                                                                 ║
  ║  Step 1: Given 192.168.1.130/26                                ║
  ║                                                                 ║
  ║  Step 2: /26 → host bits = 32-26 = 6 → block = 2^6 = 64      ║
  ║                                                                 ║
  ║  Step 3: Subnets at: .0, .64, .128, .192                      ║
  ║          130 falls in .128 block                                ║
  ║                                                                 ║
  ║  Step 4: Network    = 192.168.1.128                            ║
  ║          First host = 192.168.1.129                            ║
  ║          Last host  = 192.168.1.190   (128+64-1-2=190)        ║
  ║          Broadcast  = 192.168.1.191   (128+64-1=191)          ║
  ║          Usable     = 62 hosts                                 ║
  ║                                                                 ║
  ║  REMEMBER:                                                      ║
  ║  • Hosts = 2^(32-prefix) - 2                                  ║
  ║  • Block size = 2^(32-prefix)                                  ║
  ║  • Magic number = 256 - mask octet                             ║
  ║  • To find network: floor(IP / block) × block                 ║
  ║                                                                 ║
  ╚══════════════════════════════════════════════════════════════════╝
""")

# Quick practice
problems = [
    ('10.10.10.200/27',  'What subnet? How many hosts?'),
    ('172.16.100.50/22', 'What is the network address?'),
    ('Need 500 hosts',   'What prefix length?'),
]

print("  Practice Problems:\n")
for ip, q in problems:
    print(f"    Q: {ip} — {q}")

print("\n  Solutions:")
print("    1: /27 → block=32, 200÷32=6.25 → net=.192, hosts=30")
print("    2: /22 → mask=255.255.252.0, 100 AND 252=100 → 172.16.100.0")
print("    3: 500 hosts → need 512 (2^9) → /23")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | How many usable hosts in /26? | 2^6 - 2 = 62 (subtract network + broadcast) |
| 2 | What is VLSM? | Variable-length subnet masks. Different subnets get different sizes based on need |
| 3 | What is supernetting? | Combining multiple smaller networks into one larger prefix. Reduces routing table size |
| 4 | Given 192.168.10.70/28, find network? | /28 → block=16. 70÷16=4.375 → net=.64. Answer: 192.168.10.64 |
| 5 | Why is VLSM better than classful? | No IP waste. Assign /30 for point-to-point, /24 for LAN, /20 for data center |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Block size = 2^(32 - prefix)                          │
## │  • Usable hosts = block size - 2                         │
## │  • Magic number = 256 - mask octet value                 │
## │  • VLSM: allocate largest subnets first                  │
## │  • Supernetting: aggregate routes, shrink routing tables  │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Routing** — Static, dynamic, RIP, OSPF, BGP